# Skyline Enrollments — Summary Statistics
**Module 01 · Lesson 1-3**

Loads `skyline_enrollments.csv` from the Lesson 1.2 folder and computes
summary statistics appropriate for each column's scale of measurement.

## Scales of Measurement

| Column | Scale of Measurement | Rationale |
|---|---|---|
| `enrollment_id` | **Nominal** | Unique record identifier — labels only, no meaningful order or arithmetic. |
| `course_name` | **Nominal** | Categorical course titles; no inherent ranking between courses. |
| `enrollment_year` | **Interval** | Calendar years have equal spacing (2023→2024 = 2024→2025), but there is no true zero year, so ratios ("twice the year") are meaningless. |
| `final_grade` | **Ordinal** | Letter grades (A > B > C > D > F) are ordered, but the gaps between adjacent grades are not guaranteed to be equal. |
| `hours_studied` | **Ratio** | Continuous numeric with a true zero (0 h = no study time at all); ratios are fully meaningful. |
| `completion_status` | **Nominal** | Unordered categories (Completed / In Progress / Dropped); no natural ranking exists among them. |

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

lesson12_dir = Path(
    "~/northstar-coursework/module-01-foundations-of-analytics-and-statistics"
    "/lesson-1-2-data-types-and-measurement-scales"
).expanduser()

df = pd.read_csv(lesson12_dir / "skyline_enrollments.csv")
print(f"Shape: {df.shape}")
df.head(10)

Shape: (100, 6)


,enrollment_id,course_name,enrollment_year,final_grade,hours_studied,completion_status
0,ENR-100000,Intro to Analytics,2024,B,22.4,Completed
1,ENR-100001,Intro to Analytics,2025,B,24.0,Completed
2,ENR-100002,SQL Basics,2022,A,28.4,Completed
3,ENR-100003,Intro to Analytics,2025,C,45.8,Completed
4,ENR-100004,Statistics 101,2026,C,28.9,Completed
5,ENR-100005,SQL Basics,2024,C,25.0,Completed
6,ENR-100006,Intro to Analytics,2023,B,47.7,Dropped
7,ENR-100007,Python for Beginners,2024,B,34.2,Completed
8,ENR-100008,Statistics 101,2024,B,21.2,Completed
9,ENR-100009,Python for Beginners,2023,C,18.7,Completed


## 1  Nominal Columns — `enrollment_id`, `course_name`, `completion_status`

Appropriate statistics: **count**, **unique count**, and **mode**.
Mean and median are not meaningful for unordered categories.

In [2]:
nominal_cols = ["enrollment_id", "course_name", "completion_status"]

rows = []
for col in nominal_cols:
    mode_val  = df[col].mode()[0]
    mode_freq = (df[col] == mode_val).sum()
    rows.append({
        "column":       col,
        "count":        df[col].count(),
        "unique_count": df[col].nunique(),
        "mode":         mode_val,
        "mode_freq":    mode_freq,
    })

pd.DataFrame(rows).set_index("column")

,count,unique_count,mode,mode_freq
column,,,,
enrollment_id,100,100,ENR-100000,1
course_name,100,5,Intro to Analytics,31
completion_status,100,3,Completed,79


## 2  Ordinal Column — `final_grade`

Appropriate statistics: **count**, **mode**, and **median** (via numeric encoding).
The mean is not formally appropriate — see the note below.

In [3]:
grade_order   = {"A": 5, "B": 4, "C": 3, "D": 2, "F": 1}
enc_to_label  = {v: k for k, v in grade_order.items()}
grade_numeric = df["final_grade"].map(grade_order)

count_g  = df["final_grade"].count()
mode_g   = df["final_grade"].mode()[0]
mode_gn  = (df["final_grade"] == mode_g).sum()
median_enc = grade_numeric.median()
median_g   = enc_to_label.get(int(median_enc), f"between grades (encoded {median_enc})")

print(f"count : {count_g}")
print(f"mode  : {mode_g}  (n={mode_gn})")
print(f"median: {median_g}  (encoded = {median_enc})")
print()
print("Grade distribution:")
print(df["final_grade"].value_counts().reindex(["A","B","C","D","F"]))

count : 100
mode  : C  (n=40)
median: C  (encoded = 3.5)

Grade distribution:
final_grade
A    12
B    38
C    40
D     2
F     8
Name: count, dtype: int64


In [4]:
# Mean computed for reference — see Markdown note below for the caveat
mean_grade_numeric = grade_numeric.mean()
print(f"Mean (numeric encoding A=5…F=1): {mean_grade_numeric:.3f}")
print(f"Nearest label: {enc_to_label[round(mean_grade_numeric)]}")

Mean (numeric encoding A=5…F=1): 3.440
Nearest label: C


### 📝 Caveat on Reporting the Mean of `final_grade`

Computing a mean for `final_grade` requires mapping letter grades to numbers
(A = 5, B = 4, C = 3, D = 2, F = 1).  That mapping embeds an assumption that the
*distance* between each adjacent pair of grades is identical — that improving from
a D to a C represents the same magnitude of change as improving from a B to an A.
Nothing in the grading system guarantees that, and a different mapping (e.g.,
A = 4.0, B = 3.0, C = 2.0, D = 1.0, F = 0.0) would produce a different mean.

**Stakeholder caveat:** Any mean grade figure should be accompanied by a disclosure
that the numeric scale was arbitrarily chosen and that the result is sensitive to
that choice.  A more defensible summary for stakeholders is the **mode** (most
common grade) together with the **full grade distribution** (count per category),
which require no interval-equality assumptions.

## 3  Interval Column — `enrollment_year`

Appropriate statistics: **mean**, **median**, **mode**, **standard deviation**, **IQR**.
A p95 or ratio-based stat (e.g. coefficient of variation) would be misleading because
`enrollment_year` has no true zero.

In [5]:
col = "enrollment_year"
y   = df[col]

q1y, q3y = y.quantile(0.25), y.quantile(0.75)

summary_year = pd.Series({
    "count":   y.count(),
    "mean":    round(y.mean(), 3),
    "median":  y.median(),
    "mode":    y.mode()[0],
    "std_dev": round(y.std(ddof=1), 3),
    "Q1":      q1y,
    "Q3":      q3y,
    "IQR":     round(q3y - q1y, 3),
}, name=col)

summary_year

count       100.000
mean       2024.220
median     2024.000
mode       2025.000
std_dev       1.353
Q1         2023.000
Q3         2025.000
IQR           2.000
Name: enrollment_year, dtype: float64

## 4  Ratio Column — `hours_studied`

All arithmetic operations are valid (true zero exists).  Appropriate statistics:
**mean**, **median**, **mode**, **standard deviation**, **IQR**, and **p95**.

In [6]:
col = "hours_studied"
h   = df[col]

q1h, q3h = h.quantile(0.25), h.quantile(0.75)

summary_hours = pd.Series({
    "count":   h.count(),
    "mean":    round(h.mean(), 3),
    "median":  h.median(),
    "mode":    h.mode()[0],
    "std_dev": round(h.std(ddof=1), 3),
    "Q1":      q1h,
    "Q3":      q3h,
    "IQR":     round(q3h - q1h, 3),
    "p95":     round(h.quantile(0.95), 3),
}, name=col)

summary_hours

count      100.000
mean        29.960
median      29.750
mode        28.900
std_dev      7.765
Q1          24.975
Q3          34.375
IQR          9.400
p95         42.140
Name: hours_studied, dtype: float64

In [7]:
mean_h = h.mean()
med_h  = h.median()
print(f"mean   = {mean_h:.2f} h")
print(f"median = {med_h:.2f} h")
print(f"mean − median = {mean_h - med_h:+.2f} h  ({100*(mean_h-med_h)/med_h:+.1f}% relative to median)")

mean   = 29.96 h
median = 29.75 h
mean − median = +0.21 h  (+0.7% relative to median)


### 📝 Mean vs. Median for `hours_studied`

The mean and median for `hours_studied` are very close to one another, suggesting the
distribution is **roughly symmetric** with no strong skew.  Unlike a right-skewed
distribution (where a long upper tail drags the mean well above the median), the two
measures of central tendency converge here, meaning either figure gives a fair
representation of the "typical" student's study time.  The mean is appropriate to
report in this case.